# Criando um conjunto de testes
Na teoria, criar um conjunto de testes é simples: escolher algumas instâncias aleatórias, geralmente 20% do conjunto de dados e defini-las

In [3]:
import numpy as np
import pandas as pd

In [4]:
def shuffle_and_split_data(data, test_ratio):
    shuffle_indices = np.random.permutation(len(data))
    test_set_size = int(len(data)*test_ratio)
    test_indices = shuffle_indices[:test_set_size]
    train_indices = shuffle_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

## Carregando Dados

In [5]:
housing = pd.read_csv("datasets/housing/housing.csv")
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Criando conjuntos de Treino e Teste

In [6]:
train_set, test_set = shuffle_and_split_data(housing, 0.2)

In [7]:
display(train_set.head())
print(len(train_set))

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
6995,-117.71,34.15,17.0,17715.0,2370.0,7665.0,2312.0,7.9068,349100.0,INLAND
13387,-118.15,34.20,46.0,1505.0,261.0,857.0,269.0,4.5000,184200.0,INLAND
14316,-118.56,34.23,36.0,3215.0,529.0,1710.0,539.0,5.5126,248400.0,<1H OCEAN
8288,-118.25,34.02,50.0,180.0,89.0,356.0,76.0,2.1944,158300.0,<1H OCEAN
11254,-118.28,34.17,22.0,2664.0,651.0,1553.0,629.0,3.6354,256300.0,<1H OCEAN


16512


In [8]:
display(test_set.head())
print(len(test_set))

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
11999,-122.21,37.78,46.0,2239.0,508.0,1390.0,569.0,2.7352,137300.0,NEAR BAY
11359,-120.86,37.50,34.0,4272.0,996.0,2916.0,962.0,1.9829,82800.0,INLAND
9744,-117.60,33.43,21.0,3951.0,562.0,1392.0,543.0,5.1439,414000.0,NEAR OCEAN
1944,-117.97,33.68,26.0,1616.0,292.0,700.0,241.0,5.5105,232100.0,<1H OCEAN
15442,-121.69,38.16,46.0,2292.0,472.0,970.0,431.0,2.2888,94900.0,INLAND


4128


Isso funciona mas, ao executar o programa novamente, um novo conjunto de testes diferente vai ser gerado e assim, ao longo do tempo o seu algoritmo acabaria vendo todo o conjunto de dados.

Uma solução aparente seria salvar o conjunto de testes na primeira execução e depois só carregar isso em execuções posteriores. Ou então, 
gerar números aleatórios com ``random.seed()`` andes de chamar ``random.permutation()`` e garantir que ele sempre gere os mesmos índices embaralhados.

Entretanto, essas duas abordagens quebram quando você altera as instâncias do conjunto de dados.

<br>

- Se você salva o conjunto de testes na primeira execução:
    
    <table>
        <tr>
            <td>
            <strong>1ª execução (100 instâncias):</strong><br>
            Treino: [0, 1, 2, ... 79]<br>
            Teste: [80, 81, ... 99] <em>← salvo em disco</em><br>
            </td>
        </tr>
    </table> 
        
    Agora o dataset é atualizado para 120 instâncias. Você carrega o teste salvo — as mesmas instâncias de antes. OK, sem contaminação do conjunto de treino.

    - O problema: As 20 novas instâncias (100 a 119) ficam todas no treino automaticamente.

    - Com o tempo, o modelo nunca vai ser testado em dados novos. Assim:
    <p align="center"><i>O teste fica "envelhecido" e pode deixar de ser representativo.</i></p>

    - Além disso, se o dataset for completamente substituído (não apenas expandido), os índices salvos podem não fazer mais sentido nenhum.

<br>

- Se você aplica uma ``seed``:

    - A seed garante que o embaralhamento seja sempre igual para o mesmo conjunto de dados. Mas se o dataset mudar (novas linhas adicionadas), o embaralhamento produz índices diferentes, porque agora existem mais elementos. Portanto:
        <p align="center"><i>Instâncias que estavam no treino podem acabar no teste, e vice-versa.</i></p>

    - Pense assim:
        <p align="center"><i>O teste existe para simular o mundo real, dados que o modelo nunca viu. Se uma instância estava no treino antes, e depois de um update ela migra pro teste, você está avaliando o modelo com dados que ele já conhece. O resultado parece bom, mas é uma ilusão.
        </i></p>
    
    - A sequência do problema:

        <table>
            <tr>
                <td>
                <strong>1ª execução (100 instâncias):</strong><br>
                Treino: instâncias [0 a 79]<br>
                Teste:  instâncias [80 a 99]<br>
                <br>
                <strong>Dataset atualizado (120 instâncias) + seed fixa:</strong><br>
                Treino: instâncias [3, 7, 15, 82...] ← embaralhamento novo <br>
                Teste:  instâncias [91, 4, 67...]  ← instância 4 estava no treino antes!<br>
                </td>
            </tr>
        </table> 

### Usando `hash` do ID para criação do Conjunto de Teste 

Um `hash` é uma função que transforma qualquer valor em um número fixo, sempre o mesmo para uma mesma entrada.

In [11]:
from zlib import crc32

print(crc32(np.int64(42))) # → sempre retorna 1322774185
print(crc32(np.int64(43)))   # → sempre retorna 3715953633
print(crc32(np.int64(42)))   # → sempre retorna 1322774185 (mesmo resultado!)

227844599
3242107241
227844599


O `hash` é:
- **Determinístico**: o mesmo ID sempre gera o mesmo hash
- **Distribuído**: os valores resultantes são espalhados de forma uniforme
- **Irreversível**: não dá pra saber o ID original a partir do hash

### Como isso é usado para o split?
O hash gera um número entre 0 e 2³². A ideia é:

<p align="center"><i>Se o hash do ID cabe nos 20% menores valores possíveis, a instância vai para o teste.</i></p>

In [ ]:
from zlib import crc32

def is_id_in_test_set(id, test_ratio):
    return crc32(np.int64(id)) < test_ratio * 2**32 #bool
    
def split_data_with_id_hash(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: is_id_in_test_set(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]

Como o *dataset* que estamos usando não tem uma coluna de *id*, a solução mais simples é usar o índice da linha como *id*

In [14]:
housing_with_id = housing.reset_index() # adiciona uma coluna 'index'

Assim, contruímos nosso conjuntos *train_set* e *test_set*

In [15]:
train_set , test_set = split_data_with_id_hash(housing_with_id, 0.2, 'index')

In [17]:
train_set

,index,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
3,3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
6,6,-122.25,37.84,52.0,2535.0,489.0,1094.0,514.0,3.6591,299200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...,...
20635,20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [18]:
test_set

,index,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
2,2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
5,5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY
12,12,-122.26,37.85,52.0,2491.0,474.0,1098.0,468.0,3.0750,213500.0,NEAR BAY
16,16,-122.27,37.85,52.0,1966.0,347.0,793.0,331.0,2.7750,152500.0,NEAR BAY
23,23,-122.27,37.84,52.0,1688.0,337.0,853.0,325.0,2.1806,99700.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...,...
20615,20615,-121.54,39.08,23.0,1076.0,216.0,724.0,197.0,2.3598,57500.0,INLAND
20617,20617,-121.53,39.06,20.0,561.0,109.0,308.0,114.0,3.3021,70800.0,INLAND
20622,20622,-121.44,39.00,20.0,755.0,147.0,457.0,157.0,2.4167,67000.0,INLAND
20626,20626,-121.43,39.18,36.0,1124.0,184.0,504.0,171.0,2.1667,93800.0,INLAND


### Identificador com Latitude e Longitude
Podemos usar outras features para produzir um id exclusivo

In [21]:
housing_with_id["id"] = housing["longitude"] * 1000 + housing["latitude"]
train_set, test_set = split_data_with_id_hash(housing_with_id, 0.2, "id")

In [26]:
train_set.head(5)

,index,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,id
0,0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,-122192.12
1,1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,-122182.14
2,2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,-122202.15
3,3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,-122212.15
4,4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,-122212.15


In [27]:
test_set.head(5)

,index,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,id
59,59,-122.29,37.82,2.0,158.0,43.0,94.0,57.0,2.5625,60000.0,NEAR BAY,-122252.18
60,60,-122.29,37.83,52.0,1121.0,211.0,554.0,187.0,3.3929,75700.0,NEAR BAY,-122252.17
61,61,-122.29,37.82,49.0,135.0,29.0,86.0,23.0,6.1183,75000.0,NEAR BAY,-122252.18
62,62,-122.29,37.81,50.0,760.0,190.0,377.0,122.0,0.9011,86100.0,NEAR BAY,-122252.19
67,67,-122.29,37.80,52.0,1027.0,244.0,492.0,147.0,2.6094,81300.0,NEAR BAY,-122252.20
